# `semantic.v_measure` — view

Thin view over Gold. No logic beyond shaping.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [0]:
-- THE MEASURE VIEW. The same 445 facts, once per measure, so one dashboard filter can
-- switch the whole page between total return, growth and income.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_measure
COMMENT 'Every ticker at every horizon on three measures: total return, growth and income'
AS
WITH spy_income AS (
  -- Income is the one measure with no per-span index column, so the index's own row is used.
  SELECT horizon_years, income_return AS spy_income
  FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance
  WHERE ticker = 'SPY'
),
base AS (
  SELECT f.horizon_years, f.ticker,
         f.total_return, f.price_return, f.income_return,
         f.beat_index, f.out_grew_index, si.spy_income,
         d.trust_name, d.entity_type, d.manager, d.management_group, d.status
  FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance f
  JOIN `index-vs-trust-pipeline`.gold.dim_ticker d ON d.ticker_key = f.ticker_key
  JOIN spy_income si ON si.horizon_years = f.horizon_years
),
-- Long, not wide: one row per measure lets a single filter switch the whole page.
-- beats_index comes from the fact's stored flags, which compare each trust against the
-- index over that trust's OWN span. Comparing against the index's own row instead would
-- disagree with the published beat rate wherever a delisted trust has a shorter span.
long AS (
  SELECT horizon_years, ticker, trust_name, entity_type, manager, management_group, status,
         'total return' AS measure, total_return AS value, beat_index AS beats_index FROM base
  UNION ALL
  SELECT horizon_years, ticker, trust_name, entity_type, manager, management_group, status,
         'growth', price_return, out_grew_index FROM base
  UNION ALL
  SELECT horizon_years, ticker, trust_name, entity_type, manager, management_group, status,
         'income', income_return, income_return > spy_income FROM base
)
SELECT horizon_years, measure, ticker, trust_name, entity_type, manager, management_group, status,
       value,
       ROUND(100 * value, 1) AS value_pct,
       beats_index,
       -- Display rank only. Ranking never filters the fact.
       RANK() OVER (PARTITION BY horizon_years, measure ORDER BY value DESC) AS rank_in_measure
FROM long;

## Verification

Expected: the view resolves and returns rows. Counts are in 
`
specs/04_semantic/dashboard.md
`
.

In [0]:
SELECT COUNT(*) AS rows
FROM `index-vs-trust-pipeline`.semantic.v_measure;